# 1. imports and install required libraries

In [1]:
! pip install openai-whisper
! pip install yt-dlp
! pip install ffmpeg-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=36cad33a6483379b38da2d34690a033c4ba3e18cb2cf3942f68ad59aed6473f0
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.3 MB/s eta 0:00:00


In [2]:
!pip install -U yt-dlp

In [15]:
import os
import json
import whisper
import torch
from datetime import datetime
print(torch.cuda.is_available())
RAW_DIR = "raw"

True


# 2. Video Download

In [4]:
os.makedirs("raw", exist_ok=True)

urls = [
"https://www.youtube.com/watch?v=oeqPrUmVz-o"
]
failed = []
for url in urls:
  try:
    print(f"Downloading:{url}")
    !yt-dlp \
      --format "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/mp4" \
      --merge-output-format mp4 \
      --write-info-json \
      --write-thumbnail \
      --convert-thumbnails jpg \
      --output "raw/%(id)s/%(id)s.%(ext)s" \
      {url}
    print(f"Finished: {url}\n")
  except Exception as e:
    print(f"Error with {url}: {e}\n")
    failed.append(url)
print(f"Failed videos:",failed)

Downloading:https://www.youtube.com/watch?v=oeqPrUmVz-o
[youtube] Extracting URL: https://www.youtube.com/watch?v=oeqPrUmVz-o
[youtube] oeqPrUmVz-o: Downloading webpage
[youtube] oeqPrUmVz-o: Downloading android vr player API JSON
[info] oeqPrUmVz-o: Downloading 1 format(s): 397+140
[info] Downloading video thumbnail 44 ...
[info] Video Thumbnail 44 does not exist
[info] Downloading video thumbnail 43 ...
[info] Video Thumbnail 43 does not exist
[info] Downloading video thumbnail 42 ...
[info] Video Thumbnail 42 does not exist
[info] Downloading video thumbnail 41 ...
[info] Video Thumbnail 41 does not exist
[info] Downloading video thumbnail 40 ...
[info] Video Thumbnail 40 does not exist
[info] Downloading video thumbnail 39 ...
[info] Video Thumbnail 39 does not exist
[info] Downloading video thumbnail 38 ...
[info] Writing video thumbnail 38 to: raw/oeqPrUmVz-o/oeqPrUmVz-o.webp
[info] Writing video metadata as JSON to: raw/oeqPrUmVz-o/oeqPrUmVz-o.info.json
[ThumbnailsConvertor] Con

In [5]:
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR,folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            old_path = os.path.join(folder_path, file)

            if file.endswith(".mp4"):
                os.rename(old_path, os.path.join(folder_path, "video.mp4"))

            elif file.endswith(".info.json"):
                os.rename(old_path, os.path.join(folder_path, "info.json"))

            elif file.endswith(".jpg"):
                os.rename(old_path, os.path.join(folder_path, "thumbnail.jpg"))

# 3.Metadata Extraction

In [6]:
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    if os.path.isdir(folder_path):

        for file in os.listdir(folder_path):
            if file.endswith("info.json"):

                info_path = os.path.join(folder_path, file)

                with open(info_path, "r") as f:
                    data = json.load(f)

                # 1. title.txt
                title = data.get("title", "")
                with open(os.path.join(folder_path, "title.txt"), "w") as f:
                    f.write(title)

                # 2. description.txt
                description = data.get("description", "")
                with open(os.path.join(folder_path, "description.txt"), "w") as f:
                    f.write(description)

                # 3. meta.json
                meta = {
                    "video_id": data.get("id"),
                    "title": title,
                    "duration_seconds": data.get("duration"),
                    "upload_date": data.get("upload_date"),
                    "channel_id": data.get("channel_id"),
                    "channel_name": data.get("channel"),
                    "category": data.get("categories"),
                    "tags": data.get("tags"),
                    "view_count": data.get("view_count"),
                    "like_count": data.get("like_count"),
                    "language": data.get("language"),
                    "thumbnail_url": data.get("thumbnail")
                }

                with open(os.path.join(folder_path, "meta.json"), "w") as f:
                    json.dump(meta, f, indent=2)

                print(f"Processed metadata for {folder}")

Processed metadata for oeqPrUmVz-o


# 4. Frame Extraction

In [7]:
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    if os.path.isdir(folder_path):

        video_path = os.path.join(folder_path, "video.mp4")
        frames_path = os.path.join(folder_path, "frames")

        if os.path.exists(video_path):
            print(f"Extracting frames for {folder}")

            os.makedirs(frames_path, exist_ok=True)

            !ffmpeg -i {video_path} -vf fps=1 -q:v 2 {frames_path}/frame_%05d.jpg

            print(f"Done {folder}\n")

Extracting frames for oeqPrUmVz-o
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enabl

## 4.1 Frame Index

In [8]:
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    if os.path.isdir(folder_path):

        frames_path = os.path.join(folder_path, "frames")

        if os.path.exists(frames_path):
            frames = sorted(os.listdir(frames_path))

            index = []
            for i, frame in enumerate(frames):
                index.append({
                    "frame_file": frame,
                    "timestamp_sec": float(i + 1)  # fps = 1
                })

            with open(os.path.join(folder_path, "frames_index.json"), "w") as f:
                json.dump(index, f, indent=2)

            print(f"Created index for {folder}")

Created index for oeqPrUmVz-o


# Frame Filtering (apply after extraction, before storage)
# Remove frames that are clearly unusable:

* **Blurry frames**: compute Laplacian variance — discard if below threshold (e.g. < 100)
* **Dark frames:** compute mean pixel brightness — discard if mean < 30
* **Near-duplicate frames:** compare consecutive frames with perceptual hash — discard if similarity > 0.95

Do not overfilter. Keep a record of which frames were discarded and why in a frames_discarded.json file. You may need them later.

In [9]:
# TODO: implement Frame Filtering

# 5. Audio Extraction

In [10]:
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    if os.path.isdir(folder_path):

        video_path = os.path.join(folder_path, "video.mp4")
        audio_path = os.path.join(folder_path, "audio.wav")

        if os.path.exists(video_path):
            print(f"Extracting audio for {folder}")

            !ffmpeg -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {audio_path}

            print(f"Done {folder}\n")
        else:
            print(f"No video found in {folder}")

Extracting audio for oeqPrUmVz-o
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable

# 6. Transcription

In [11]:
model = whisper.load_model("small")
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    if os.path.isdir(folder_path):
        audio_path = os.path.join(folder_path, "audio.wav")
        transcript_path = os.path.join(folder_path, "transcript.json")

        if os.path.exists(audio_path):
            print(f"Transcribing {folder}...")

            result = model.transcribe(
                audio_path,
                language="en",
                word_timestamps=True
            )

            with open(transcript_path, "w") as f:
                json.dump(result["segments"], f, indent=2)

            print(f"Done {folder}\n")
        else:
            print(f"No audio found in {folder}")

100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 78.4MiB/s]


Transcribing oeqPrUmVz-o...
Done oeqPrUmVz-o



#7. alignment

In [13]:
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    frames_index_path = os.path.join(folder_path, "frames_index.json")
    transcript_path = os.path.join(folder_path, "transcript.json")

    if os.path.exists(frames_index_path) and os.path.exists(transcript_path):

        print(f"Processing {folder}")

        with open(frames_index_path) as f:
            frames = json.load(f)

        with open(transcript_path) as f:
            segments = json.load(f)

        def get_text(t):
            for seg in segments:
                if seg["start"] <= t <= seg["end"]:
                    return seg["text"]
            return ""

        pairs = []

        for item in frames:
            t = item["timestamp_sec"]
            text = get_text(t)

            pairs.append({
                "frame": item["frame_file"],
                "text": text
            })

        print(pairs[:5])

Processing oeqPrUmVz-o
[{'frame': 'frame_00001.jpg', 'text': ''}, {'frame': 'frame_00002.jpg', 'text': " Mr. Jobs, you're a bright and influential man."}, {'frame': 'frame_00003.jpg', 'text': " Mr. Jobs, you're a bright and influential man."}, {'frame': 'frame_00004.jpg', 'text': " Mr. Jobs, you're a bright and influential man."}, {'frame': 'frame_00005.jpg', 'text': ''}]


# Manual Annotation Fields

In [16]:
ANNOTATOR = "zakria"
for folder in os.listdir(RAW_DIR):
    folder_path = os.path.join(RAW_DIR, folder)

    if os.path.isdir(folder_path):

        annotation_path = os.path.join(folder_path, "annotation.json")

        if os.path.exists(annotation_path):
            continue

        annotation = {
            "video_id": folder,
            "annotator": ANNOTATOR,
            "annotation_date": datetime.today().strftime('%Y-%m-%d'),
            "thumbnail_label": None,
            "thumbnail_label_confidence": None,
            "notes": ""
        }

        with open(annotation_path, "w") as f:
            json.dump(annotation, f, indent=2)

        print(f"Created annotation for {folder}")

Created annotation for oeqPrUmVz-o


In [12]:
# !zip -r data.zip data/

	zip warning: name not matched: data/

zip error: Nothing to do! (try: zip -r data.zip . -i data/)
